In [18]:
%pip install pandas seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
import pandas as pd
import seaborn as sns
import os
# chargement du jeu de données brut
titanic = sns.load_dataset("titanic")


In [20]:
titanic["age"] = titanic.groupby(["class", "sex"], observed=False)["age"].transform(lambda x: x.fillna(x.median()))

In [21]:
port_majoritaire = titanic["embark_town"].mode()[0]
titanic["embark_town"] = titanic["embark_town"].fillna(port_majoritaire)

In [22]:
titanic = titanic.drop(columns=["deck"], errors="ignore")

In [23]:
doublons = titanic.duplicated().sum()
print(f"Nombre de doublons : {doublons}")
# Décision : on ne supprime pas si l'on considère qu'il s'agit de personnes différentes partageant le même tarif/classe,
# ou alors on supprime si l'on estime que chaque ligne doit être unique :
titanic = titanic.drop_duplicates()

Nombre de doublons : 118


In [24]:
# Suppression des doublons
colonnes_inutiles = ["alive", "pclass", "embarked"]
titanic = titanic.drop(columns=colonnes_inutiles)

# Conversion des colonnes texte en catégories (gain mémoire)
titanic["sex"] = titanic["sex"].astype("category")
titanic["embark_town"] = titanic["embark_town"].astype("category")
titanic.info()

<class 'pandas.DataFrame'>
Index: 773 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     773 non-null    int64   
 1   sex          773 non-null    category
 2   age          773 non-null    float64 
 3   sibsp        773 non-null    int64   
 4   parch        773 non-null    int64   
 5   fare         773 non-null    float64 
 6   class        773 non-null    category
 7   who          773 non-null    str     
 8   adult_male   773 non-null    bool    
 9   embark_town  773 non-null    category
 10  alone        773 non-null    bool    
dtypes: bool(2), category(3), float64(2), int64(3), str(1)
memory usage: 46.2 KB


In [28]:
billets_anormaux = titanic[(titanic["fare"] == 0) | (titanic["fare"] > 300)]
print(billets_anormaux[["fare", "class", "sex",]])

         fare   class     sex
179    0.0000   Third    male
258  512.3292   First  female
263    0.0000   First    male
271    0.0000   Third    male
277    0.0000  Second    male
302    0.0000   Third    male
597    0.0000   Third    male
679  512.3292   First    male
737  512.3292   First    male
806    0.0000   First    male
822    0.0000   First    male


**Décision sur les tarifs hors normes :**
Les billets à 0 $ (personnel/invités) et le billet record à 512,33 $ (suites de luxe) sont conservés car ils correspondent à des réalités historiques documentées et non à des erreurs de saisie.

In [29]:
# Création du dossier cible s'il n'existe pas encore
os.makedirs("data", exist_ok=True)
# Export du jeu de données nettoyé
titanic.to_csv("data/titanic_propre.csv", index=False)

Le journal de nettoyage (Cellule Markdown)Crée une cellule Markdown dans ton notebook pour résumer tes arbitrages sous forme de tableau :   Markdown### Journal de nettoyage

| Quoi | Pourquoi | Lignes / Impact |
| :--- | :--- | :--- |
| `deck` supprimée | 77 % de valeurs manquantes | 891 |
| `age` imputé | Remplacé par la médiane du groupe classe x sexe | 177 |
| `embark_town` imputé | Remplacé par le port majoritaire (Southampton) | 2 |
| `alive`, `pclass`, `embarked` supprimées | Doublons d'informations avec `survived`,